# Advanced Usage

This notebook demonstrates advanced usage patterns for trainlib:

- **Custom data formats** — register your own data format handlers
- **Custom model loaders** — integrate non-standard architectures
- **Model loading options** — quantization, dtype, device mapping
- **Callback system** — hook into training events
- **Logging backends** — TensorBoard, Weights & Biases, MLflow
- **Export pipelines** — merge, save, GGUF conversion, Hub push
- **Config system** — YAML configs with overrides
- **Distributed training** — DDP, FSDP, DeepSpeed strategies
- **End-to-end workflows** — complete training pipelines

> **Note:** Code cells show exact API usage but won't execute without GPU/models. This is a guided reference notebook.

## Custom Data Formats

Register custom data format handlers for domain-specific preprocessing. The `@register_format` decorator lets you define how your raw data maps to training samples.

In [ ]:
from trainlib.data import register_format

@register_format("medical-qa")
def format_medical_qa(sample):
    """Format medical Q&A data for training."""
    return {
        "text": f"Question: {sample['question']}\nContext: {sample['context']}\nAnswer: {sample['answer']}"
    }

# Use the custom format
import trainlib

state = trainlib.finetune(
    model="meta-llama/Llama-3.1-8B",
    dataset="data/medical_qa.jsonl",
    method="lora",
    format="medical-qa",
)

## Custom Model Loaders

Integrate custom model architectures with the `@register_model` decorator. Return a `ModelResult` containing your model, tokenizer, and metadata.

In [ ]:
from trainlib.models import register_model

@register_model("my-custom-arch")
def load_my_model(name_or_path, **kwargs):
    """Load a custom model architecture."""
    from trainlib.models.loader import ModelResult
    
    model = MyCustomModel.from_pretrained(name_or_path)
    tokenizer = AutoTokenizer.from_pretrained(name_or_path)
    
    return ModelResult(
        model=model,
        tokenizer=tokenizer,
        name=name_or_path,
    )

## Model Loading Options

Control how models are loaded with quantization, dtype, device mapping, and PEFT adapters.

In [ ]:
from trainlib.models import load_model

# Standard loading
result = load_model("meta-llama/Llama-3.1-8B")

# 4-bit quantized
result = load_model("meta-llama/Llama-3.1-70B", quantization="4bit")

# Custom dtype and device mapping
result = load_model(
    "meta-llama/Llama-3.1-8B",
    dtype="bfloat16",
    device_map="auto",
    trust_remote_code=True,
)

# Apply LoRA adapters
from trainlib.models.peft import apply_lora

result = apply_lora(
    result,
    rank=32,
    alpha=64,
    dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

## Callback System

Hook into training events with decorators. Available events: `train_start`, `train_end`, `epoch_start`, `epoch_end`, `step_start`, `step_end`, `eval_start`, `eval_end`, `checkpoint_saved`, `error`.

In [ ]:
from trainlib.trainer import on

@on("train_start")
def on_start(state):
    print(f"Training started — {state.num_epochs} epochs")

@on("step_end")
def log_step(state):
    if state.global_step % 100 == 0:
        loss = state.metrics.get("loss", "N/A")
        print(f"Step {state.global_step}: loss={loss}")

@on("epoch_end")
def on_epoch(state):
    print(f"Epoch {state.epoch} complete")

@on("train_end")
def on_finish(state):
    print(f"Training complete — {state.global_step} total steps")

## Custom Callback Manager

Create your own callback manager for advanced control flow.

In [ ]:
from trainlib.trainer import CallbackManager

manager = CallbackManager()

@manager.on("step_end")
def custom_logging(state):
    if state.metrics.get("loss", float("inf")) < 0.1:
        print("Loss target reached!")
        state.stop_training()

## Logging Backends

Configure logging to console, TensorBoard, Weights & Biases, or MLflow. Multiple backends can be used simultaneously.

In [ ]:
from trainlib.config.schema import LoggingConfig

# Console only (default)
logging = LoggingConfig(backends=["console"])

# TensorBoard
logging = LoggingConfig(
    backends=["console", "tensorboard"],
    project="my-experiment",
)

# Weights & Biases (requires pip install trainlib[wandb])
logging = LoggingConfig(
    backends=["console", "wandb"],
    project="my-project",
    run_name="lora-8b-v1",
)

# MLflow (requires pip install trainlib[mlflow])
logging = LoggingConfig(
    backends=["console", "mlflow"],
    project="my-project",
)

# All backends at once
logging = LoggingConfig(
    backends=["console", "tensorboard", "wandb"],
    project="my-project",
    log_every_n_steps=10,
)

## Export Pipeline

Complete workflow from training to deployment: merge adapters, save with metadata, convert to GGUF, and push to Hugging Face Hub.

In [ ]:
from trainlib.export import merge, save, push_to_hub, to_gguf

# 1. Merge LoRA adapters into base model
merge("output/lora-checkpoint", save_to="output/merged-model")

# 2. Save with metadata
from trainlib.models import load_model
result = load_model("output/merged-model")
save(
    result.model,
    result.tokenizer,
    output_dir="output/final-model",
    metadata={"recipe": "finetune", "method": "lora", "base_model": "Llama-3.1-8B"},
)

# 3. Convert to GGUF for llama.cpp / Ollama
to_gguf("output/merged-model", output="output/model.gguf", quantization="Q4_K_M")

# 4. Push to Hugging Face Hub
push_to_hub("output/merged-model", repo="username/my-model")

## GGUF Quantization Options

Common quantization presets for GGUF conversion. Lower quantization = smaller file, faster inference, slight quality loss.

In [ ]:
from trainlib.export import to_gguf

# Common quantization options
to_gguf("model/", output="model-q4.gguf", quantization="Q4_K_M")   # Good balance
to_gguf("model/", output="model-q5.gguf", quantization="Q5_K_M")   # Higher quality
to_gguf("model/", output="model-q8.gguf", quantization="Q8_0")     # Highest quality

## Config System

Load training configurations from YAML files with optional runtime overrides.

In [ ]:
from trainlib.config import load_config, validate_config

# Load from YAML
config = load_config("configs/examples/lora_finetune.yaml")

# Load with overrides
config = load_config(
    "configs/examples/lora_finetune.yaml",
    overrides=["model.name=mistralai/Mistral-7B-v0.3", "trainer.num_epochs=5"],
)

# Validate config
validate_config(config)

# Use with recipe
import trainlib
state = trainlib.finetune(config=config)

## Distributed Training

Scale to multiple GPUs or nodes with DDP, FSDP, or DeepSpeed strategies.

In [ ]:
from trainlib.config.schema import TrainConfig, ModelConfig, DataConfig, TrainerConfig

# DDP (Data Distributed Parallel)
config = TrainConfig(
    recipe="finetune",
    method="lora",
    model=ModelConfig(name="meta-llama/Llama-3.1-8B"),
    data=DataConfig(path="data/train.jsonl", format="alpaca"),
    trainer=TrainerConfig(strategy="ddp", batch_size=4),
)

# FSDP (Fully Sharded Data Parallel) — for large models
config = TrainConfig(
    recipe="finetune",
    method="full",
    model=ModelConfig(name="meta-llama/Llama-3.1-8B"),
    data=DataConfig(path="data/train.jsonl", format="alpaca"),
    trainer=TrainerConfig(strategy="fsdp", batch_size=2),
)

# DeepSpeed (requires pip install trainlib[deepspeed])
config = TrainConfig(
    recipe="pretrain",
    model=ModelConfig(name="meta-llama/Llama-3.1-8B"),
    data=DataConfig(path="data/corpus/", format="text"),
    trainer=TrainerConfig(strategy="deepspeed", batch_size=2),
)

## End-to-End Workflow

Complete training pipeline: fine-tune, merge adapters, align with preferences, evaluate, and push to Hub.

In [ ]:
import trainlib
from trainlib.export import merge, push_to_hub

# Step 1: Fine-tune
state = trainlib.finetune(
    model="meta-llama/Llama-3.1-8B",
    dataset="data/instructions.jsonl",
    method="lora",
    format="alpaca",
    num_epochs=3,
)

# Step 2: Merge adapters
merge(f"{state.metrics.get('output_dir', 'output')}", save_to="output/merged")

# Step 3: Align with preferences
state = trainlib.align(
    model="output/merged",
    dataset="data/preferences.jsonl",
    method="dpo",
    format="preference",
)

# Step 4: Evaluate
results = trainlib.evaluate(
    model="output/merged",
    dataset=eval_data,
    metrics=["loss", "perplexity"],
)
print(f"Final perplexity: {results['perplexity']:.2f}")

# Step 5: Push to Hub
push_to_hub("output/merged", repo="username/my-llm")